# Reconstructing the Webis-Editorials-16 corpus from XMI

This notebook reconstructs the Webis-Editorials-16 corpus into a JSONL format suitable for training ModernBERT for my Master's thesis.

Each document will be represented as one JSON record with:

- `id`: document identifier
- `text`: full article text
- `spans`: annotated argumentative/appeal units

Each span contains:

- `start`: character start offset
- `end`: character end offset
- `label`: annotation label
- `text`: exact text covered by the span

The reconstruction uses the original XMI files rather than the annotated TXT files, because the XMI contains both:

1. the full article text in `cas:Sofa/@sofaString`
2. the annotation offsets and labels in `ArgumentativeDiscourseUnit`

This avoids offset mismatches caused by formatting differences in TXT exports. 

_Note: this second version of the reconstruction exists precisely due to offset missmatches created by initially parsing the TXT_

In [1]:
from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET
import json
from collections import Counter, defaultdict

# Path to the uploaded Webis corpus ZIP.
# If your file is in a different location, change this path.
zip_path = Path("corpus-webis-editorials-16.zip")

# Directory where we will extract the ZIP.
extract_dir = Path("corpus_webis_extracted")

print("ZIP exists:", zip_path.exists())
print("ZIP path:", zip_path.resolve() if zip_path.exists() else "not found")

ZIP exists: True
ZIP path: C:\Users\andre\OneDrive\Desktop\Master\Semester 2\Thesis\corpus-webis-editorials-16.zip


## Extract the corpus ZIP and inspect the XMI files

The Webis corpus is saved as a ZIP file. In this step, I extract it into a local folder and search for all `.xmi` files.

Each XMI file should correspond to one annotated editorial document.

In [2]:
# Extract the ZIP only if it has not already been extracted.
if not extract_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)
    print("ZIP extracted.")
else:
    print("Extraction directory already exists.")

# Find all XMI files inside the extracted corpus.
xmi_files = sorted(extract_dir.rglob("*.xmi"))

print("Number of XMI files found:", len(xmi_files))

# Show a few example paths.
for path in xmi_files[:10]:
    print(path)

ZIP extracted.
Number of XMI files found: 600
corpus_webis_extracted\corpus-webis-editorials-16\annotated-xmi\split-by-portal-final\aljazeera\006-afghanistanatcrucialjuncture.xmi
corpus_webis_extracted\corpus-webis-editorials-16\annotated-xmi\split-by-portal-final\aljazeera\007-afghanistanconferencewonrock.xmi
corpus_webis_extracted\corpus-webis-editorials-16\annotated-xmi\split-by-portal-final\aljazeera\008-afghanistanreadydrawdown.xmi
corpus_webis_extracted\corpus-webis-editorials-16\annotated-xmi\split-by-portal-final\aljazeera\009-aftercharliehebdoattackisla.xmi
corpus_webis_extracted\corpus-webis-editorials-16\annotated-xmi\split-by-portal-final\aljazeera\012-anotherclimatechangesummitd.xmi
corpus_webis_extracted\corpus-webis-editorials-16\annotated-xmi\split-by-portal-final\aljazeera\017-argentinagrowthatwhatprice.xmi
corpus_webis_extracted\corpus-webis-editorials-16\annotated-xmi\split-by-portal-final\aljazeera\018-armeniacancountrussiaanymo.xmi
corpus_webis_extracted\corpus-web

## Inspect one XMI file

Before reconstructing the full corpus, we inspect one XMI file to confirm where the document text and annotations are stored.

The full article text should be stored in `cas:Sofa` as `sofaString`.

The annotated units should be stored as `ArgumentativeDiscourseUnit` elements with:

- `begin`
- `end`
- `unitType`

In [4]:
# Inspect the first XMI file to understand structure

example_xmi_path = xmi_files[0]

print("Example XMI path:")
print(example_xmi_path)

tree = ET.parse(example_xmi_path)
root = tree.getroot()

print("\nRoot tag:")
print(root.tag)

print("\nFirst 20 XML elements and their attributes:")
for i, elem in enumerate(root.iter()):
    if i >= 20:
        break
    print("TAG:", elem.tag)
    print("ATTRIBUTES:", elem.attrib)
    print()

Example XMI path:
corpus_webis_extracted\corpus-webis-editorials-16\annotated-xmi\split-by-portal-final\aljazeera\006-afghanistanatcrucialjuncture.xmi

Root tag:
{http://www.omg.org/XMI}XMI

First 20 XML elements and their attributes:
TAG: {http://www.omg.org/XMI}XMI
ATTRIBUTES: {'{http://www.omg.org/XMI}version': '2.0'}

TAG: {http:///uima/cas.ecore}NULL
ATTRIBUTES: {'{http://www.omg.org/XMI}id': '0'}

TAG: {http:///uima/tcas.ecore}DocumentAnnotation
ATTRIBUTES: {'{http://www.omg.org/XMI}id': '8', 'sofa': '1', 'begin': '0', 'end': '6070', 'language': 'x-unspecified'}

TAG: {http:///de/aitools/ie/uima/type/argumentation.ecore}ArgumentativeDiscourseUnit
ATTRIBUTES: {'{http://www.omg.org/XMI}id': '13', 'sofa': '1', 'begin': '0', 'end': '72', 'unitType': 'anecdote'}

TAG: {http:///de/aitools/ie/uima/type/argumentation.ecore}ArgumentativeDiscourseUnit
ATTRIBUTES: {'{http://www.omg.org/XMI}id': '18', 'sofa': '1', 'begin': '74', 'end': '239', 'unitType': 'assumption'}

TAG: {http:///de/aitoo

## Extract the full document text from `cas:Sofa`

In UIMA XMI files, the full document text is stored in the `cas:Sofa` element as the `sofaString` attribute.

The annotation offsets (`begin` and `end`) should refer to character positions inside this text.

In [5]:
# Find the cas:Sofa element and extract the full text.

sofa = root.find(".//{http:///uima/cas.ecore}Sofa")

print("Found Sofa:", sofa is not None)

if sofa is not None:
    document_text = sofa.attrib["sofaString"]

    print("Document text length:", len(document_text))
    print("\nFirst 1000 characters:")
    print(document_text[:1000])

Found Sofa: True
Document text length: 6070

First 1000 characters:
On Thursday, the so-called London Conference on Afghanistan will convene. Of all the high-profile international conferences that have been held on Afghanistan over the years - in Bonn, Tokyo, and Istanbul - this comes at a crucial juncture.

At the end of October, British forces lowered the flag at Camp Bastion - from where they had engaged in the most intense fighting the British army had seen since the Falklands War - and handed over the base to Afghan forces. Just over a month later, the Taliban mounted a 14-hour attack on the very same site, killing six Afghan soldiers. To the north, in the capital Kabul, insurgents have pulled off a dozen attacks in the space of the last few weeks, striking at diplomats, NGOs, and US contractors. What explains this spate of violence, and is it a portent for the future of Afghanistan?

The International Security Assistance Force (ISAF), a US-led group that includes NATO members and

## Extract annotation spans from one XMI file

Now we extract the `ArgumentativeDiscourseUnit` annotations from one example XMI file.

For each unit, we collect:

- `start`: the `begin` offset
- `end`: the `end` offset
- `label`: the `unitType`
- `text`: the substring from the full document text

At this stage, we keep all labels, including `no-unit`, only to inspect the data. Later, we will exclude `no-unit` from the training spans.

In [6]:
# Extract all ArgumentativeDiscourseUnit spans from the example XMI file.

ARG_UNIT_TAG = "{http:///de/aitools/ie/uima/type/argumentation.ecore}ArgumentativeDiscourseUnit"

example_spans = []

for elem in root.iter(ARG_UNIT_TAG):
    start = int(elem.attrib["begin"])
    end = int(elem.attrib["end"])
    label = elem.attrib["unitType"]
    span_text = document_text[start:end]

    example_spans.append({
        "start": start,
        "end": end,
        "label": label,
        "text": span_text,
    })

print("Number of spans in example document:", len(example_spans))
print("Label counts:", Counter(span["label"] for span in example_spans))

print("\nFirst 5 spans:")
for span in example_spans[:5]:
    print(span["start"], span["end"], span["label"])
    print(repr(span["text"][:200]))
    print()

Number of spans in example document: 65
Label counts: Counter({'assumption': 30, 'no-unit': 14, 'anecdote': 12, 'statistics': 4, 'testimony': 4, 'common-ground': 1})

First 5 spans:
0 72 anecdote
'On Thursday, the so-called London Conference on Afghanistan will convene'

74 239 assumption
'Of all the high-profile international conferences that have been held on Afghanistan over the years - in Bonn, Tokyo, and Istanbul - this comes at a crucial juncture'

242 465 anecdote
'At the end of October, British forces lowered the flag at Camp Bastion - from where they had engaged in the most intense fighting the British army had seen since the Falklands War - and handed over th'

467 579 anecdote
'Just over a month later, the Taliban mounted a 14-hour attack on the very same site, killing six Afghan soldiers'

581 743 anecdote
'To the north, in the capital Kabul, insurgents have pulled off a dozen attacks in the space of the last few weeks, striking at diplomats, NGOs, and US contractors'



## Reconstruct one document record

Next, we reconstruct one document into the target JSON structure.

The target format is:

```json
{
  "id": "document_id",
  "text": "full article text",
  "spans": [
    {
      "start": 0,
      "end": 72,
      "label": "anecdote",
      "text": "exact span text"
    }
  ]
} 
```

In [7]:
# Reconstruct one document record from the example XMI file.

TARGET_LABELS = {
    "anecdote",
    "testimony",
    "common-ground",
    "assumption",
    "statistics",
    "other",
}

def get_document_id_from_path(xmi_path):
    return xmi_path.stem


def reconstruct_record_from_xmi(xmi_path):
    tree = ET.parse(xmi_path)
    root = tree.getroot()

    sofa = root.find(".//{http:///uima/cas.ecore}Sofa")
    if sofa is None:
        raise ValueError(f"No cas:Sofa element found in {xmi_path}")

    text = sofa.attrib["sofaString"]

    spans = []

    for elem in root.iter(ARG_UNIT_TAG):
        start = int(elem.attrib["begin"])
        end = int(elem.attrib["end"])
        label = elem.attrib["unitType"]

        # Exclude no-unit from the training/evaluation span list.
        if label == "no-unit":
            continue

        if label not in TARGET_LABELS:
            raise ValueError(f"Unexpected label '{label}' in {xmi_path}")

        span_text = text[start:end]

        spans.append({
            "start": start,
            "end": end,
            "label": label,
            "text": span_text,
        })

    spans = sorted(spans, key=lambda span: (span["start"], span["end"]))

    return {
        "id": get_document_id_from_path(xmi_path),
        "text": text,
        "spans": spans,
    }


example_record = reconstruct_record_from_xmi(example_xmi_path)

print("Document ID:", example_record["id"])
print("Text length:", len(example_record["text"]))
print("Number of spans excluding no-unit:", len(example_record["spans"]))
print("Label counts:", Counter(span["label"] for span in example_record["spans"]))

print("\nFirst 3 reconstructed spans:")
for span in example_record["spans"][:3]:
    print(span["start"], span["end"], span["label"])
    print(repr(span["text"]))
    print()

Document ID: 006-afghanistanatcrucialjuncture
Text length: 6070
Number of spans excluding no-unit: 51
Label counts: Counter({'assumption': 30, 'anecdote': 12, 'statistics': 4, 'testimony': 4, 'common-ground': 1})

First 3 reconstructed spans:
0 72 anecdote
'On Thursday, the so-called London Conference on Afghanistan will convene'

74 239 assumption
'Of all the high-profile international conferences that have been held on Afghanistan over the years - in Bonn, Tokyo, and Istanbul - this comes at a crucial juncture'

242 465 anecdote
'At the end of October, British forces lowered the flag at Camp Bastion - from where they had engaged in the most intense fighting the British army had seen since the Falklands War - and handed over the base to Afghan forces'



## Validate one reconstructed document

Before reconstructing the whole corpus, we validate one document.

The validation checks that:

- each span has valid integer offsets
- `start < end`
- the span lies inside the document text
- `text[start:end]` exactly matches `span["text"]`
- spans are sorted
- spans do not overlap

In [8]:
def validate_record(record):
    """
    Validate one reconstructed document record.
    
    Returns a list of error messages.
    If the list is empty, the record is valid.
    """
    errors = []
    
    text = record["text"]
    spans = record["spans"]
    
    previous_end = -1
    
    for i, span in enumerate(spans):
        start = span["start"]
        end = span["end"]
        label = span["label"]
        span_text = span["text"]
        
        if not isinstance(start, int):
            errors.append(f"Span {i}: start is not an integer")
            continue
            
        if not isinstance(end, int):
            errors.append(f"Span {i}: end is not an integer")
            continue
        
        if start < 0:
            errors.append(f"Span {i}: start is negative")
        
        if end > len(text):
            errors.append(f"Span {i}: end is beyond document length")
        
        if start >= end:
            errors.append(f"Span {i}: start >= end")
        
        if label not in TARGET_LABELS:
            errors.append(f"Span {i}: unexpected label {label}")
        
        if 0 <= start < end <= len(text):
            extracted_text = text[start:end]
            if extracted_text != span_text:
                errors.append(
                    f"Span {i}: span text mismatch. "
                    f"Expected {repr(extracted_text)}, got {repr(span_text)}"
                )
        
        if start < previous_end:
            errors.append(
                f"Span {i}: overlapping or unsorted span. "
                f"Previous end: {previous_end}, current start: {start}"
            )
        
        previous_end = end
    
    return errors


example_errors = validate_record(example_record)

print("Number of validation errors:", len(example_errors))

if example_errors:
    print("\nFirst validation errors:")
    for error in example_errors[:10]:
        print(error)
else:
    print("Example record is valid.")

Number of validation errors: 0
Example record is valid.


## Reconstruct the official train, validation, and test splits

The Webis corpus already contains an official evaluation split in:

`annotated-xmi/split-for-evaluation-final/`

This folder contains three subfolders:

- `train`
- `validation`
- `test`

In this step, we locate those folders and count how many XMI files are available in each split.

In [13]:
# Locate the official train/validation/test split folders.

root_dir = extract_dir / "corpus-webis-editorials-16"

split_dir = root_dir / "annotated-xmi" / "split-for-evaluation-final"

train_dir = split_dir / "training"
validation_dir = split_dir / "validation"
test_dir = split_dir / "test"

print("Root directory exists:", root_dir.exists())
print("Split directory exists:", split_dir.exists())
print("Train directory exists:", train_dir.exists())
print("Validation directory exists:", validation_dir.exists())
print("Test directory exists:", test_dir.exists())

train_xmi_files = sorted(train_dir.glob("*.xmi"))
validation_xmi_files = sorted(validation_dir.glob("*.xmi"))
test_xmi_files = sorted(test_dir.glob("*.xmi"))

print("\nNumber of train XMI files:", len(train_xmi_files))
print("Number of validation XMI files:", len(validation_xmi_files))
print("Number of test XMI files:", len(test_xmi_files))

print("\nFirst 5 train files:")
for path in train_xmi_files[:5]:
    print(path.name)

Root directory exists: True
Split directory exists: True
Train directory exists: True
Validation directory exists: True
Test directory exists: True

Number of train XMI files: 180
Number of validation XMI files: 60
Number of test XMI files: 60

First 5 train files:
001-2015beyondobamanewcongressneed.xmi
002-2015willamericacontinueitsslow.xmi
003-2015willnewcongressgetseriousa.xmi
005-actorartistphilanthropistmymot.xmi
006-afghanistanatcrucialjuncture.xmi


## Reconstruct records for each split

Now that the official split folders have been located, we reconstruct the records for each split separately.

Each split is reconstructed from its own XMI files:

- training XMI files → training records
- validation XMI files → validation records
- test XMI files → test records

After reconstruction, we validate every record in each split.

In [14]:
def reconstruct_split(xmi_paths):
    """
    Reconstruct a list of JSON-style records from a list of XMI file paths.
    """
    split_records = []
    split_errors = {}

    for xmi_path in xmi_paths:
        record = reconstruct_record_from_xmi(xmi_path)
        split_records.append(record)

        errors = validate_record(record)
        if errors:
            split_errors[record["id"]] = errors

    return split_records, split_errors


train_records, train_errors = reconstruct_split(train_xmi_files)
validation_records, validation_errors = reconstruct_split(validation_xmi_files)
test_records, test_errors = reconstruct_split(test_xmi_files)

print("Train records:", len(train_records))
print("Validation records:", len(validation_records))
print("Test records:", len(test_records))

print("\nDocuments with validation errors:")
print("Train:", len(train_errors))
print("Validation:", len(validation_errors))
print("Test:", len(test_errors))

if train_errors or validation_errors or test_errors:
    print("\nThere are validation errors. Inspect them before saving.")
else:
    print("\nAll split records are valid.")

Train records: 180
Validation records: 60
Test records: 60

Documents with validation errors:
Train: 0
Validation: 0
Test: 0

All split records are valid.


## Inspect split statistics

Before saving the reconstructed corpus, we inspect basic statistics for each split:

- number of documents
- total number of labeled spans
- label distribution

This helps confirm that the reconstructed corpus contains the expected six labels and that `no-unit` has been excluded.

In [15]:
def summarize_split(split_name, split_records):
    """
    Print basic statistics for one reconstructed split.
    """
    label_counts = Counter()
    total_spans = 0

    for record in split_records:
        total_spans += len(record["spans"])
        label_counts.update(span["label"] for span in record["spans"])

    print(f"{split_name}")
    print("-" * len(split_name))
    print("Documents:", len(split_records))
    print("Total labeled spans:", total_spans)
    print("Label counts:")

    for label, count in label_counts.most_common():
        print(f"  {label}: {count}")

    print()


summarize_split("Training split", train_records)
summarize_split("Validation split", validation_records)
summarize_split("Test split", test_records)

Training split
--------------
Documents: 180
Total labeled spans: 8498
Label counts:
  assumption: 5603
  anecdote: 1624
  testimony: 738
  statistics: 272
  common-ground: 171
  other: 90

Validation split
----------------
Documents: 60
Total labeled spans: 2971
Label counts:
  assumption: 2186
  anecdote: 493
  testimony: 137
  statistics: 68
  other: 46
  common-ground: 41

Test split
----------
Documents: 60
Total labeled spans: 2844
Label counts:
  assumption: 2003
  anecdote: 486
  testimony: 214
  statistics: 81
  other: 31
  common-ground: 29



## Save reconstructed splits as JSONL files

Finally, we save the reconstructed corpus into three JSONL files:

- `webis_editorials_train.jsonl`
- `webis_editorials_validation.jsonl`
- `webis_editorials_test.jsonl`

Each line contains one document record. This format is convenient for later conversion to BIO format and model fine-tuning.

In [16]:
output_dir = Path("reconstructed_webis_jsonl")
output_dir.mkdir(exist_ok=True)

train_output_path = output_dir / "webis_editorials_train.jsonl"
validation_output_path = output_dir / "webis_editorials_validation.jsonl"
test_output_path = output_dir / "webis_editorials_test.jsonl"


def save_jsonl(records, path):
    """
    Save a list of records as a JSONL file.
    Each line is one JSON object.
    """
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


save_jsonl(train_records, train_output_path)
save_jsonl(validation_records, validation_output_path)
save_jsonl(test_records, test_output_path)

print("Saved files:")
print(train_output_path)
print(validation_output_path)
print(test_output_path)

print("\nFile sizes:")
print("Train:", train_output_path.stat().st_size, "bytes")
print("Validation:", validation_output_path.stat().st_size, "bytes")
print("Test:", test_output_path.stat().st_size, "bytes")

Saved files:
reconstructed_webis_jsonl\webis_editorials_train.jsonl
reconstructed_webis_jsonl\webis_editorials_validation.jsonl
reconstructed_webis_jsonl\webis_editorials_test.jsonl

File sizes:
Train: 2317229 bytes
Validation: 786622 bytes
Test: 756329 bytes


## Reload saved JSONL files and verify them

After saving the reconstructed splits, we reload the JSONL files from disk.

This final check confirms that the saved files can be read correctly and that they still contain the expected number of documents and spans.

In [17]:
def load_jsonl(path):
    """
    Load a JSONL file into a list of records.
    """
    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    return records


reloaded_train_records = load_jsonl(train_output_path)
reloaded_validation_records = load_jsonl(validation_output_path)
reloaded_test_records = load_jsonl(test_output_path)

print("Reloaded train records:", len(reloaded_train_records))
print("Reloaded validation records:", len(reloaded_validation_records))
print("Reloaded test records:", len(reloaded_test_records))

print("\nReloaded split statistics:")
summarize_split("Reloaded training split", reloaded_train_records)
summarize_split("Reloaded validation split", reloaded_validation_records)
summarize_split("Reloaded test split", reloaded_test_records)

Reloaded train records: 180
Reloaded validation records: 60
Reloaded test records: 60

Reloaded split statistics:
Reloaded training split
-----------------------
Documents: 180
Total labeled spans: 8498
Label counts:
  assumption: 5603
  anecdote: 1624
  testimony: 738
  statistics: 272
  common-ground: 171
  other: 90

Reloaded validation split
-------------------------
Documents: 60
Total labeled spans: 2971
Label counts:
  assumption: 2186
  anecdote: 493
  testimony: 137
  statistics: 68
  other: 46
  common-ground: 41

Reloaded test split
-------------------
Documents: 60
Total labeled spans: 2844
Label counts:
  assumption: 2003
  anecdote: 486
  testimony: 214
  statistics: 81
  other: 31
  common-ground: 29



## Final output

The Webis-Editorials-16 corpus has been reconstructed from the official XMI files and saved as three JSONL files.

The final files are stored in:

`reconstructed_webis_jsonl/`

Files:

- `webis_editorials_train.jsonl`
- `webis_editorials_validation.jsonl`
- `webis_editorials_test.jsonl`

These files contain span-level annotations and can be used as input for the next preprocessing step: conversion to BIO format for token classification.

In [18]:
print("Final reconstructed files:")
print(train_output_path)
print(validation_output_path)
print(test_output_path)

print("\nExample record keys:")
print(reloaded_train_records[0].keys())

print("\nExample document ID:")
print(reloaded_train_records[0]["id"])

print("\nText preview:")
print(reloaded_train_records[0]["text"][:500])

print("\nFirst 3 spans:")
for span in reloaded_train_records[0]["spans"][:3]:
    print(span)

Final reconstructed files:
reconstructed_webis_jsonl\webis_editorials_train.jsonl
reconstructed_webis_jsonl\webis_editorials_validation.jsonl
reconstructed_webis_jsonl\webis_editorials_test.jsonl

Example record keys:
dict_keys(['id', 'text', 'spans'])

Example document ID:
001-2015beyondobamanewcongressneed

Text preview:
In the film, "Girl Interrupted," Winona Ryder plays an 18-year-old who enters a mental institution for what is diagnosed as borderline personality disorder. The year is 1967 and the country is in turmoil over Vietnam and civil rights. While lying on her bed one night and watching TV, she sees a news report about a demonstration. The narrator says something that might apply to today's turmoil: "We live in a time of doubt. The institutions we once trusted no longer seem reliable."

As 2014 ends, t

First 3 spans:
{'start': 0, 'end': 155, 'label': 'anecdote', 'text': 'In the film, "Girl Interrupted," Winona Ryder plays an 18-year-old who enters a mental institution for 